In [1]:
##ai agent class day2 assignment
##Movie Expert Agent

import openai, json

client = openai.OpenAI()

messages = []


In [2]:
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


##/movies에서 인기 영화를 가져옵니다.
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

##/movies/:id에서 영화 정보를 가져옵니다.
def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

##/movies/:id/credits에서 출연진 및 제작진을 가져옵니다.
def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    response.raise_for_status()
    return json.dumps(response.json(), ensure_ascii=False)

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": """
            Get the list of currently popular movies.

            Use this function when the user asks:
            - What movies are popular now?
            - Popular movies
            - Trending movies
            - 현재 인기 영화
            - 인기 영화 추천
            """,
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": """
            Get detailed information about a movie by its movie ID.

            Use this function when the user asks:
            - What movie is ID 550?
            - Tell me about movie 550
            - 영화 550 정보 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": """
            Get cast and crew information for a movie.

            Use this function when the user asks:
            - Who starred in movie 550?
            - Cast of movie 550
            - 출연진 알려줘
            - 감독 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [4]:
SYSTEM_PROMPT = """
You are a Movie Expert Agent.

You have access to three tools:

1. get_popular_movies()
   - Returns the list of currently popular movies.

2. get_movie_details(id)
   - Returns detailed information for a movie.

3. get_movie_credits(id)
   - Returns cast and crew information for a movie.

Rules:
- If the user asks about popular movies, call get_popular_movies.
- If the user asks about a specific movie ID, call get_movie_details.
- If the user asks who acted in a movie or asks for cast/crew, call get_movie_credits.
- Always use tools when movie information is required.
- After receiving tool results, answer the user in Korean.
"""

In [5]:
messages.append({
    "role": "system",
    "content": SYSTEM_PROMPT
})

In [9]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            
            result = function_to_run(**arguments)

           # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
            })

        call_ai()

    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [10]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with {}
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Obsession**
   - 개요: 소중한 사람의 마음을 얻기 위해 신비한 "원 위시 윌로우"를 깨뜨린 후, 절망적인 로맨티스트는 자신의 소원이 이루어지지만 일부 욕망은 어두운 대가를 치르게 된다는 이야기입니다.
   - 평점: 7.9
   - 개봉일: 2026-05-13
   - ![포스터](https://image.tmdb.org/t/p/w780/2G249T4Sgu8gXIZpaXWnxHYYNQV.jpg)

2. **Peddi**
   - 개요: 1980년대 안드라 프라데시 농촌에서, 한 열정적인 촌민이 스포츠를 통해 지역 사회를 단결시켜 강력한 라이벌에 맞섰다는 이야기입니다.
   - 평점: 6.3
   - 개봉일: 2026-06-03
   - ![포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **Lee Cronin's The Mummy**
   - 개요: 한 기자의 어린 딸이 사막에서 흔적 없이 사라지고, 8년 후 가족이 재회하지만 그 재회는 악몽으로 변하게 된다는 이야기입니다.
   - 평점: 8.1
   - 개봉일: 2026-04-15
   - ![포스터](https://image.tmdb.org/t/p/w780/1q308iixueCU4pFtSYugNOevtNx.jpg)

4. **The Mandalorian and Grogu**
   - 개요: 악당 제국이 무너지면서, 새로운 공화국이 자신의 안위를 위해 전설적인 맨달로리안 현상금 사냥꾼과 그의 어린 제자 그로구의 도움을 요청하게 되는 이야기입니다.
   - 평점: 6.8
   - 개봉일: 2026-05-20
   - ![포스터](https://image.tmdb.org/t/p/w780/5Vi8dSauVwH1HOsiZceD